'''TASK 3·Cleaning Data
Objective: Demonstrate professional-level data cleaning skills by taking a deliberately messy dataset and systematically transforming it into a clean, analysis-ready dataset. Document every decision.'''

'''1.Load dataset and produce a "data quality report": count of nulls per column, duplicate rows, data type issues, value range anomalies'''

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("C:/Users/Ayush Singh/OIBSIB/L1-pre-DataCleaning.csv")
#copying data rn, later for comparison
df_before=df.copy()

In [3]:
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print("\nNull Counts")
print(df.isnull().sum())
print("\nDuplicate Rows")
print(f"Count:{df.duplicated().sum()}")
print("\nData Types")
print(df.dtypes)
print(f"\nPrice Range:{df['Price'].min()} to {df['Price'].max()}")
print(f"\nQuantity Range:{df['Quantity'].min()} to {df['Quantity'].max()}")
print(f"\nUnique Categories:{df['Category'].unique()}")
print(f"\nSample Prices:{df['Price'].unique()[:10]}")

Total Rows: 103
Total Columns: 11

Null Counts
ID                 0
 Customer_Name     0
Order_ID           0
Order_Date         0
Product            0
Category           8
Quantity           5
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64

Duplicate Rows
Count:1

Data Types
ID                  int64
 Customer_Name        str
Order_ID              str
Order_Date            str
Product               str
Category              str
Quantity              str
Price                 str
Payment_Method        str
Status                str
Total             float64
dtype: object

Price Range:-100 to four hundred

Quantity Range:-2 to 5

Unique Categories:<StringArray>
[       'Home', 'Electronics',      'Sports',       'Books',      'sports',
    'Clothing',  'electronic', 'ELECTRONICS', 'electronics',           nan]
Length: 10, dtype: str

Sample Prices:<StringArray>
[    '38',    'abd', '389.05', '233.92', '552.51', '122.06', '978.63',
 '587.6

'''2.Duplicate removal: identify and remove duplicate rows; document how many were removed'''

In [4]:
d_before=df.duplicated().sum()
df.drop_duplicates(inplace=True)
d_removed=d_before - df.duplicated().sum()
print(f"\nDuplicates Removed:{d_removed}")


Duplicates Removed:1


'''3.Standardisation: normalise inconsistent formatting'''

In [5]:
df['Category']= df['Category'].astype(str).str.strip().str.title()
category_map= {'Electronic': 'Electronics', 'Sports': 'Sports', 'Home': 'Home'}
df['Category']= df['Category'].replace(category_map)

df['Payment_Method']= df['Payment_Method'].astype(str).str.strip().str.title()
df['Status']= df['Status'].astype(str).str.strip().str.title()

df['Price']= df['Price'].astype(str).str.replace('$', '').str.replace(',', '')
df['Price']= df['Price'].replace({'four hundred': '400', 'Three Hundred': '300'}, regex=False)
df['Price']= pd.to_numeric(df['Price'], errors='coerce')

df['Quantity']= pd.to_numeric(df['Quantity'], errors='coerce')
df['Order_Date']= pd.to_datetime(df['Order_Date'], errors='coerce', format='mixed')

In [6]:
df

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3.0,38.00,Cash On Delivery,Shipped,114.000
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2.0,NaN,Paypal,Processing,NaN
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,Sports,1.0,389.05,Paypal,Delivered,389.050
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5.0,233.92,Paypal,Processing,1169.600
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1.0,552.51,Cash On Delivery,Processing,552.510
...,...,...,...,...,...,...,...,...,...,...,...
97,197,Customer_197,ORD-79139,2025-06-23,Blender,Home,1.0,160.16,Paypal,Cancelled,160.160
98,198,Customer_198,ORD-14608,2025-07-27,Vacuum,NaN,2.0,497.01,Cash On Delivery,Shipped,994.020
99,199,Customer_199,ORD-82922,2025-01-22,Blender,Home,5.0,372.28,Credit Card,Shipped,1861.400
100,175,Customer_175,ORD-56651,2025-02-24,Headphones,Electronics,1.0,111.36,Credit Card,Processing,77.952


'''4.Missing data handling: choose an appropriate strategy for each column and justify each choice in a markdown cell'''

In [7]:
imputation_log=[]
# Strategy:Median for 'Price'
median_price= df['Price'].median()
df['Price']=df['Price'].fillna(median_price)
imputation_log.append(f"Price: Filled {df['Price'].isnull().sum()} missing values with Median ${median_price:.2f} to resist skew from extreme outliers.")
# Strategy:Median for 'Quantity'
median_qty= df['Quantity'].median()
df['Quantity']=df['Quantity'].fillna(median_qty)
imputation_log.append(f"Quantity: Filled missing values with Median {median_qty} as order quantities are typically right-skewed.")
# Strategy:Mode for 'Payment_Method'
mode_payment= df['Payment_Method'].mode()[0]
df['Payment_Method']=df['Payment_Method'].fillna(mode_payment)
imputation_log.append(f"Payment_Method: Filled missing values with Mode '{mode_payment}' representing the most common payment type.")
# Strategy:Mode for 'Status'
mode_status= df['Status'].mode()[0]
df['Status']=df['Status'].fillna(mode_status)
imputation_log.append(f"Status: Filled missing values with Mode '{mode_status}' to maintain order status distribution.")
# Strategy:Forward Fill for 'Order_Date'
# df['Order_Date']=df['Order_Date'].fillna(method='ffill')
print("\nIMPUTATION LOG")
for log in imputation_log:
    print(log)


IMPUTATION LOG
Price: Filled 0 missing values with Median $531.06 to resist skew from extreme outliers.
Quantity: Filled missing values with Median 3.0 as order quantities are typically right-skewed.
Payment_Method: Filled missing values with Mode 'Cash On Delivery' representing the most common payment type.
Status: Filled missing values with Mode 'Returned' to maintain order status distribution.


'''5.Outlier detection: use IQR method or Z-score to identify outliers in numeric columns; decide and document whether to cap, remove, or retain each'''

In [8]:
n_cols=['Price', 'Quantity']
outlier_log=[]
bounds={}
outlier_counts={}
for col in n_cols:
    Q1=df[col].quantile(0.25)
    Q3=df[col].quantile(0.75)
    IQR=Q3-Q1
    l=Q1- 1.5*IQR
    u=Q3+ 1.5*IQR
    outliers= df[(df[col]<l) | (df[col]>u)]
    negative_count=len(df[df[col]<0]) if col=='Quantity' else 0
    bounds[col]= {'lower':l, 'upper':u}
    outlier_counts[col]= {'total':len(outliers), 'negative':negative_count}
    
if 'Quantity' in n_cols:
    neg_count= outlier_counts['Quantity']['negative']
    if neg_count>0:
        df=df[df['Quantity']>=0]
    if neg_count>0:
        Q1=df['Quantity'].quantile(0.25)
        Q3=df['Quantity'].quantile(0.75)
        IQR= Q3-Q1
        upper_cap= Q3+ 1.5*IQR
    else:
        upper_cap=bounds['Quantity']['upper']
    df['Quantity']= np.where(df['Quantity']> upper_cap,upper_cap,df['Quantity'])
    total_orig= outlier_counts['Quantity']['total']
    outlier_log.append(f"Quantity: Removed {neg_count} negative values. Capped high outliers at {upper_cap:.2f}.")
    
if 'Price' in n_cols:
    count= outlier_counts['Price']['total']
    l_bound= bounds['Price']['lower']
    u_bound= bounds['Price']['upper']
    
    if count>0:
        df['Price']=np.where(df['Price']<0,0,df['Price'])
        df['Price']=np.where(df['Price']>u_bound,u_bound,df['Price'])
        outlier_log.append(f"Price: Detected {count} outliers. Action: Capped to IQR bounds [${l_bound:.2f}, ${u_bound:.2f}].")
    else:
        outlier_log.append(f"Price: No outliers detected.")

print("\nOUTLIER LOG")
for log in outlier_log:
    print(log)   


OUTLIER LOG
Quantity: Removed 2 negative values. Capped high outliers at 7.62.
Price: Detected 1 outliers. Action: Capped to IQR bounds [$-344.43, $1331.84].


'''6.Data type correction: ensure all columns have the correct dtype'''

In [9]:
df['Order_ID']=df['Order_ID'].astype(str)  # ID as string
df[' Customer_Name']=df[' Customer_Name'].astype(str)
df['Price']=df['Price'].astype(float)
df['Quantity']=df['Quantity'].astype(int)
df['Order_Date']=df['Order_Date'].astype('datetime64[ns]')
df['Total']=df['Total'].astype(str)

In [10]:
df

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,38.00,Cash On Delivery,Shipped,114.0
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2,531.06,Paypal,Processing,NaN
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,Sports,1,389.05,Paypal,Delivered,389.05
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5,233.92,Paypal,Processing,1169.6
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1,552.51,Cash On Delivery,Processing,552.51
...,...,...,...,...,...,...,...,...,...,...,...
97,197,Customer_197,ORD-79139,2025-06-23,Blender,Home,1,160.16,Paypal,Cancelled,160.16
98,198,Customer_198,ORD-14608,2025-07-27,Vacuum,NaN,2,497.01,Cash On Delivery,Shipped,994.02
99,199,Customer_199,ORD-82922,2025-01-22,Blender,Home,5,372.28,Credit Card,Shipped,1861.4
100,175,Customer_175,ORD-56651,2025-02-24,Headphones,Electronics,1,111.36,Credit Card,Processing,77.952


'''7.Produce a "before vs. after" summary table: null count, duplicate count, row count, and dtype accuracy — before and after cleaning'''

In [11]:
def get_summary_stats(dataframe,label):
    return{
        'Stage':label,
        'Row_Count':len(dataframe),
        'Null_Count':dataframe.isnull().sum().sum(),
        'Duplicate_Count':dataframe.duplicated().sum(),
        'Dtype_Accuracy':"Corrected(ID=str, Date=datetime, Price=float)" if label=="After" else "Raw/Mixed"
    }
summary_before= get_summary_stats(df_before,"Before Cleaning")
summary_after= get_summary_stats(df,"After Cleaning")
summary_df=pd.DataFrame([summary_before,summary_after])

print(summary_df.to_string(index=False))

          Stage  Row_Count  Null_Count  Duplicate_Count Dtype_Accuracy
Before Cleaning        103          32                1      Raw/Mixed
 After Cleaning        100          23                0      Raw/Mixed


'''8.Save the cleaned dataset to a new CSV file'''

In [12]:
output_file='Cleaned_Ecommerce_Sales.csv'
df.to_csv(output_file,index=False)
print(f"\nCleaned dataset saved to '{output_file}'")
print(df.head())


Cleaned dataset saved to 'Cleaned_Ecommerce_Sales.csv'
    ID  Customer_Name   Order_ID Order_Date        Product     Category  \
0  100   Customer_100  ORD-41285 2024-11-22        Blender         Home   
1  101   Customer_101  ORD-35783 2025-07-05     Smartphone  Electronics   
2  102   Customer_102  ORD-84355 2024-12-23  Tennis Racket       Sports   
3  103   Customer_103  ORD-57811 2025-03-19        Science        Books   
4  104   Customer_104  ORD-93614 2025-10-20      Biography        Books   

   Quantity   Price    Payment_Method      Status   Total  
0         3   38.00  Cash On Delivery     Shipped   114.0  
1         2  531.06            Paypal  Processing     NaN  
2         1  389.05            Paypal   Delivered  389.05  
3         5  233.92            Paypal  Processing  1169.6  
4         1  552.51  Cash On Delivery  Processing  552.51  
